# LG-06: 预构建 Agent（Prebuilt Agents）

> **阶段**: LG-06 | **难度**: 中级 | **预计时长**: 2-3 小时（含 1 小时实战）

## 学习目标
- 用 `create_react_agent` 3 行代码搭建带工具调用能力的 Agent
- 理解预构建 Agent 的内部机制（ReAct 循环、should_continue 路由、步数限制）
- 掌握多 Agent 协作模式：`create_supervisor`（监督者）、`create_swarm`（群体协作）
- 建立预构建 vs 自定义 StateGraph 的决策框架
- 学会防止 Agent 的 message history 撑爆上下文窗口
- 能够自定义和扩展预构建 Agent

---

In [ ]:
# 安装依赖（如未安装请取消注释）
# !pip install -U langgraph langchain langchain-openai langgraph-supervisor langgraph-swarm

## 1. 开场引入：为什么需要预构建 Agent？

上节课我们学了 Tool——怎么定义、怎么包装、怎么缓存。但要让 LLM 调用 Tool，你需要自己搭一整张图：
- LLM 节点
- ToolNode
- 条件边（判断是否有 tool_calls）
- 循环边（ToolMessage 回到 LLM 继续思考）

如果需求就是一个标准的『LLM 思考 → 调用工具 → 观察结果 → 再思考』循环，从零写图太繁琐了。

LangGraph 提供了预构建的 Agent：`create_react_agent`。3 行代码搞定。但天下没有免费的午餐——它快，但是黑盒。今天我们要搞清楚：**它到底做了什么、什么时候用、什么时候千万别用。**

## 2. create_react_agent 快速上手

### 2.1 定义工具

先定义两个简单工具：天气查询和计算器。

In [ ]:
import math
from langchain_core.tools import tool


# ==========================================
# 工具 1：查询天气（模拟）
# ==========================================
@tool
def query_weather(city: str) -> str:
    """查询指定城市的天气。"""
    weather_db = {
        "北京": "晴，25°C，空气质量良",
        "上海": "多云，28°C，湿度65%",
        "深圳": "小雨，30°C，东南风3级",
        "杭州": "阴，22°C，空气质量优",
    }
    return weather_db.get(city, f"抱歉，暂无 {city} 的天气数据")


# ==========================================
# 工具 2：计算器
# ==========================================
@tool
def calculator(expression: str) -> str:
    """计算数学表达式，支持 + - * / sqrt pow 等。"""
    try:
        # 安全评估：只允许数学运算
        allowed_names = {
            "sqrt": math.sqrt,
            "pow": math.pow,
            "sin": math.sin,
            "cos": math.cos,
            "pi": math.pi,
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"计算结果: {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"


print("工具定义完成:")
print(f"  - {query_weather.name}: {query_weather.description}")
print(f"  - {calculator.name}: {calculator.description}")

### 2.2 3 行代码搭建 Agent

In [ ]:
import os
from langgraph.prebuilt import create_react_agent

from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage



# ==========================================

# 初始化 LLM（这里用 GPT-4o，可替换为其他模型）

# ==========================================

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o"), temperature=0)



# 工具列表

tools = [query_weather, calculator]



# ==========================================

# 3 行代码，一个 Agent 就搭好了！

# ==========================================

agent = create_react_agent(llm, tools)



print("Agent 创建成功！")

print("执行流程: LLM 思考 → 调用工具 → 观察结果 → 再思考 → 回复用户")

### 2.3 运行 Agent

In [ ]:
# ==========================================
# 运行 Agent：查询天气 + 计算
# ==========================================
user_input = "北京天气怎么样？23的平方根是多少？"

print(f"用户: {user_input}")
print("=" * 50)

result = agent.invoke({
    "messages": [HumanMessage(content=user_input)]
})

# 取最后一条 AI 消息作为回复
final_reply = result["messages"][-1].content
print(f"\nAgent 回复: {final_reply}")

**输出过程解析**：
```
用户：北京天气怎么样？23的平方根是多少？
→ LLM 思考：需要查天气 + 算平方根
→ LLM 输出 tool_calls：[query_weather(city="北京"), calculator(expression="sqrt(23)")]
→ ToolNode 执行两个工具
→ ToolMessage 返回结果
→ LLM 再次思考，整合结果
→ LLM 回复用户
```

这一切，`create_react_agent` 内部自动完成了。

## 3. 内部机制白盒化：ReAct 循环是怎么工作的？

`create_react_agent` 不是魔法——它底层也是一个 StateGraph，只是帮你搭好了。

### 3.1 核心路由逻辑：should_continue

循环终止的唯一判据：**LLM 返回的 `AIMessage` 中是否包含 `tool_calls` 字段**。

```python
# 伪代码（对应源码 langgraph/prebuilt/chat_agent_executor.py:770-798）
def should_continue(state):
    messages = state["messages"]
    last_message = messages[-1]
    
    # 如果最后一条不是 AIMessage，或者没有 tool_calls → 结束
    if not isinstance(last_message, AIMessage) or not last_message.tool_calls:
        return END  # ← 循环终止！
    
    # 否则，继续去 tools 节点执行
    return "tools"
```

**关键结论**：
- LLM 返回 `AIMessage(tool_calls=[...])` → 去 `tools` 节点执行
- LLM 返回 `AIMessage(tool_calls=[])` 或普通文本 → 结束循环

这不是 LLM "主动告诉" 程序要结束，而是程序检查 LLM 的输出格式——**没有 tool call 请求就意味着 LLM 已经给出了最终答案**。

### 3.2 步数限制：防止无限循环

预构建 Agent 内置了安全机制，防止 LLM 无限调用工具。

In [ ]:
# ==========================================
# 观察内置的 State 结构
# ==========================================
from langgraph.prebuilt.chat_agent_executor import AgentState

print("create_react_agent 的 State 结构:")
print("  messages: 对话历史（自动管理）")
print("  is_last_step: 是否是最后一步（由运行时自动管理）")
print("  remaining_steps: 剩余步数（由运行时自动管理）")
print("\n注意：这些字段由 LangGraph 运行时自动注入，无需手动设置")

# ==========================================
# 步数限制源码逻辑（伪代码）
# ==========================================
print("\n" + "=" * 60)
print("步数限制机制:")
print("=" * 60)
print("""
def _are_more_steps_needed(state, response):
    has_tool_calls = isinstance(response, AIMessage) and response.tool_calls
    remaining_steps = state.get('remaining_steps', None)
    is_last_step = state.get('is_last_step', False)
    return (
        # 情况1：is_last_step=True 且还有 tool_calls → 步数用完了但LLM还想调工具
        (remaining_steps is None and is_last_step and has_tool_calls)
        # 情况2：remaining_steps < 2 且还有 tool_calls → 再调一次就会超出限制
        or (remaining_steps is not None and remaining_steps < 2 and has_tool_calls)
    )

# 如果 _are_more_steps_needed 返回 True，LLM 输出被强制替换为:
# AIMessage(content='Sorry, need more steps to process this request.')
# 这条消息没有 tool_calls → should_continue 返回 END → 循环终止
""")

当步数用完但 LLM 还想调工具时，Agent 会强制替换 LLM 的输出为：
```
AIMessage(content="Sorry, need more steps to process this request.")
```
这条消息**没有 `tool_calls`**，所以下一轮 `should_continue` 检查到为空 → 循环结束。

**这是 LangGraph 的安全机制**：即使 LLM 无限想调工具，也会被 `remaining_steps` 或 `is_last_step` 强制截断。

### 3.3 图结构可视化

`create_react_agent` 自动帮你搭了这些：
1. `agent` 节点：调用 LLM
2. `tools` 节点：ToolNode 执行工具
3. 条件边 `should_continue`：检测 `AIMessage.tool_calls` 决定走向
4. 循环边 `tools → agent`：ToolMessage 回到 LLM 继续思考
5. `AgentState`：自动管理 `messages`、`remaining_steps`、`is_last_step`

```python
# 源码级图构建（langgraph/prebuilt/chat_agent_executor.py:800-929）
workflow = StateGraph(state_schema=state_schema or AgentState)

# 定义两个节点：agent（调用LLM）和 tools（执行工具）
workflow.add_node("agent", RunnableCallable(call_model, acall_model))
workflow.add_node("tools", tool_node)

# 设置入口：从 agent 节点开始
workflow.set_entry_point("agent")

# agent → should_continue → tools / END
workflow.add_conditional_edges("agent", should_continue, path_map=agent_paths)

# tools 执行完 → 回到 agent（除非有 return_direct 的工具）
workflow.add_edge("tools", "agent")

return workflow.compile(...)
```

In [ ]:
from IPython.display import Image, display

# 生成预构建 Agent 的图结构可视化
png_bytes = agent.get_graph().draw_mermaid_png()
display(Image(data=png_bytes))
print("上图展示了 create_react_agent 的内部图结构")
print("流程: START → agent → [有tool_calls?] → tools → agent (循环) → END")

### 3.4 State 结构

```python
# create_react_agent 的 State 等价于：
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

**就这一个字段**。简单，但也意味着：
- ❌ 你无法自定义其他状态字段（比如 `retry_count`、`user_role`、`session_id`）
- ❌ 你无法在 State 里存业务数据（比如订单号、用户偏好）
- ✅ 但如果你只需要对话历史，这就够了

## 4. 自定义 System Prompt 和配置

### 4.1 通过 prompt 参数传入系统提示词

In [ ]:
from langchain_core.messages import SystemMessage

# ==========================================
# 方式 1：直接传字符串
# ==========================================
agent_with_prompt = create_react_agent(
    llm,
    tools,
    prompt="你是一个专业的天气助手，回答要简洁，用中文回复。"
)

# 测试
result = agent_with_prompt.invoke({
    "messages": [HumanMessage(content="上海天气怎么样？")]
})
print("带 system prompt 的回复:")
print(result["messages"][-1].content)

# ==========================================
# 方式 2：传 SystemMessage 对象
# ==========================================
agent_with_system_msg = create_react_agent(
    llm,
    tools,
    prompt=SystemMessage(content="你是数学专家，所有计算必须精确到小数点后4位。")
)

result2 = agent_with_system_msg.invoke({
    "messages": [HumanMessage(content="计算 2 的 10 次方除以 7")]
})
print("\n带 SystemMessage 的回复:")
print(result2["messages"][-1].content)

### 4.2 通过 prompt 参数传函数（动态生成提示词）

当需要根据当前 State 动态生成 system prompt 时，可以传一个函数：

In [ ]:
from langchain_core.messages import BaseMessage


def dynamic_prompt(state: dict) -> list[BaseMessage]:
    """
    根据对话历史动态生成 system prompt。
    如果用户提到"紧急"，切换为紧急模式。
    """
    messages = state["messages"]
    user_text = messages[-1].content if messages else ""
    
    if "紧急" in user_text:
        system_msg = SystemMessage(
            content="🚨 紧急模式：你是应急专家，必须优先处理紧急问题，回答要快速直接。"
        )
    else:
        system_msg = SystemMessage(
            content="你是普通助手，回答要详细、友好。"
        )
    
    # 返回 system message + 原始消息
    return [system_msg] + messages


agent_dynamic = create_react_agent(llm, tools, prompt=dynamic_prompt)

# 测试普通模式
print("=== 普通模式 ===")
r1 = agent_dynamic.invoke({"messages": [HumanMessage(content="北京天气？")]})
print(r1["messages"][-1].content[:100] + "...\n")

# 测试紧急模式
print("=== 紧急模式 ===")
r2 = agent_dynamic.invoke({"messages": [HumanMessage(content="紧急！服务器宕机了怎么办？")]})
print(r2["messages"][-1].content[:100] + "...")

## 5. 添加记忆：Checkpointer 让 Agent 记住对话

预构建 Agent 默认是无状态的——每次 `invoke()` 都是独立的。要让它记住对话历史，需要传入 `checkpointer`。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# ==========================================
# 创建内存中的 Checkpointer
# ==========================================
memory = InMemorySaver()

# 编译 Agent 时传入 checkpointer
agent_with_memory = create_react_agent(llm, tools, checkpointer=memory)

# 配置 thread_id，同一个 thread_id 共享记忆
config = {"configurable": {"thread_id": "user-123"}}

print("Agent 已添加记忆能力")
print("同一个 thread_id 的多轮对话会自动保留上下文\n")

# 第一轮对话
print("=== 第 1 轮 ===")
r1 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="我叫小明")]},
    config=config
)
print(f"Agent: {r1['messages'][-1].content}\n")

# 第二轮对话（不重复自我介绍）
print("=== 第 2 轮 ===")
r2 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="我叫什么名字？")]},
    config=config
)
print(f"Agent: {r2['messages'][-1].content}\n")

# 第三轮对话（跨轮引用）
print("=== 第 3 轮 ===")
r3 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="帮我查一下杭州天气，然后计算 15 的平方根")]},
    config=config
)
print(f"Agent: {r3['messages'][-1].content}")

**关键理解**：
- `checkpointer` 负责持久化 State，让 Agent 跨调用保持状态
- `thread_id` 是记忆的标识符，相同 `thread_id` 共享同一份 State
- 每次 `invoke()` 传入的新消息会自动追加到已有历史中

## 6. 消息历史管理：防止上下文窗口爆炸

预构建 Agent 的 State 只有 `messages` 一个字段，每次 tool 调用都会把结果追加到 messages 里：

```
[HumanMessage] → [AIMessage + tool_calls] → [ToolMessage(结果)]
    → [AIMessage + tool_calls] → [ToolMessage(结果)]
    → ...（循环多次后 messages 越来越长）
```

一个 session 里如果循环 10 次，messages 可能有 30+ 条，Token 轻松过万。

### 6.1 策略 1：Message Trimming（裁剪历史）

In [ ]:
from langchain_core.messages import trim_messages


def trimming_hook(state: dict) -> list[BaseMessage]:
    """
    在调用 LLM 前自动裁剪消息历史。
    只保留最近的 4000 tokens。
    """
    messages = state["messages"]
    
    # 如果消息太多，裁剪到最近 4000 tokens
    if len(messages) > 10:
        messages = trim_messages(
            messages,
            max_tokens=4000,
            strategy="last",
            token_counter=llm,
        )
        print(f"[裁剪] 消息历史过长，已裁剪到 {len(messages)} 条")
    
    return messages


agent_trimmed = create_react_agent(
    llm,
    tools,
    prompt=trimming_hook
)

print("Agent 已配置消息裁剪")
print("当消息超过 10 条时，自动裁剪到最近 4000 tokens")

### 6.2 策略 2：Summarization（总结历史）

In [ ]:
from langchain_core.messages import RemoveMessage


def summarize_if_needed(state: dict) -> list[BaseMessage]:
    """
    当消息太长时，总结早期内容。
    保留最近 4 条消息，总结前面的。
    """
    messages = state["messages"]
    
    # 简单判断：超过 12 条就总结
    if len(messages) <= 12:
        return messages
    
    # 保留最近 4 条
    to_summarize = messages[:-4]
    recent = messages[-4:]
    
    # 用 LLM 总结早期对话
    summary_prompt = [
        SystemMessage("请用一句话总结以下对话的关键事实，保留所有重要信息。"),
        *to_summarize
    ]
    summary_response = llm.invoke(summary_prompt)
    summary = summary_response.content
    
    print(f"[总结] 早期 {len(to_summarize)} 条消息已总结为: {summary[:50]}...")
    
    # 返回：总结 + 最近消息
    return [SystemMessage(f"历史对话总结: {summary}")] + recent


agent_summary = create_react_agent(
    llm,
    tools,
    prompt=summarize_if_needed
)

print("Agent 已配置自动总结")
print("当消息超过 12 条时，早期内容会被 LLM 智能总结")

### 6.3 策略 3：Tool 结果精简（从源头控制）

In [ ]:
# ==========================================
# 精简版天气查询工具：只返回关键信息
# ==========================================
@tool
def query_weather_concise(city: str) -> str:
    """查询天气，只返回一句话关键信息。"""
    weather_db = {
        "北京": "晴25°C",
        "上海": "多云28°C",
        "深圳": "小雨30°C",
    }
    return weather_db.get(city, f"暂无{city}数据")


# ==========================================
# 精简版日志查询工具
# ==========================================
@tool
def query_logs_concise(service: str, lines: int = 10) -> str:
    """查询服务日志，只返回摘要。"""
    # 模拟：不管要多少条，只返回摘要
    return f"[{service}] 最近 {lines} 条日志: 2 条 ERROR, {lines-2} 条 INFO. 主要错误: 连接超时"


concise_tools = [query_weather_concise, calculator, query_logs_concise]
agent_concise = create_react_agent(llm, concise_tools)

print("Agent 已配置精简工具")
print("工具输出已限制长度，从源头控制 Token 消耗")

# 测试：即使要 1000 条日志，也只返回摘要
result = agent_concise.invoke({
    "messages": [HumanMessage(content="查询 api-gateway 的 1000 条日志")]
})
print(f"\n回复: {result['messages'][-1].content}")

**三种策略对比**：

| 策略 | 实现难度 | 信息损失 | 适用场景 |
|------|---------|---------|---------|
| Trimming | 低 | 中（丢掉早期消息） | 对话轮数多的场景 |
| Summarization | 中 | 低（智能压缩） | 需要保留上下文的复杂对话 |
| Tool 精简 | 低 | 可控（按需提取） | 所有场景（应该在定义 tool 时就做） |

## 7. 观察内部过程：用 stream 代替 invoke

`invoke()` 是个黑盒——只给最终结果。用 `stream()` 可以看到每一步的详细过程。

In [ ]:
# ==========================================
# 使用 stream 观察 Agent 的每一步
# ==========================================
user_input = "查一下北京和上海天气，再算 16 的平方根"

print(f"用户: {user_input}")
print("=" * 60)

step = 0
for chunk in agent.stream(
    {"messages": [HumanMessage(content=user_input)]},
    stream_mode="values"
):
    step += 1
    messages = chunk["messages"]
    last_msg = messages[-1]
    
    print(f"\n--- Step {step} ---")
    print(f"消息类型: {last_msg.type}")
    
    if last_msg.type == "ai":
        print(f"AI 内容: {last_msg.content[:80]}...")
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            print(f"Tool Calls: {[tc['name'] for tc in last_msg.tool_calls]}")
    elif last_msg.type == "tool":
        print(f"Tool 结果: {last_msg.content[:80]}...")
    
    print(f"当前消息总数: {len(messages)}")

print("\n" + "=" * 60)
print(f"最终回复: {messages[-1].content}")

## 8. 多 Agent 预构建模式

除了单 Agent 的 `create_react_agent`，LangGraph 生态还提供了两个专门用于**多 Agent 协作**的预构建库。

> **注意**：`create_supervisor` 和 `create_swarm` 不在核心 `langgraph` 包中，需要额外安装 `pip install langgraph-supervisor langgraph-swarm`。

### 8.1 create_supervisor（监督者模式）

来自 `langgraph-supervisor`，适合**层级化管理**多个专业化 Agent。

**核心思想**：Supervisor 本身也是一个 `create_react_agent`，但它的 `tools` 里包含了一组 **handoff tools**（交接工具），每个 handoff tool 对应一个 sub-agent。

In [ ]:
# ==========================================
# create_supervisor 演示（代码结构展示）
# ==========================================

print("create_supervisor 核心源码分析:")
print("=" * 60)
print("""
# 1. Supervisor 的本质：就是一个 create_react_agent
# langgraph_supervisor/supervisor.py:425-434
supervisor_agent = create_react_agent(
    name=supervisor_name,
    model=model,
    tools=tool_node,          # ← 关键：tools 里包含了所有 sub-agent 的 handoff_tools
    prompt=prompt,
    state_schema=supervisor_schema,
    ...
)

# 2. Handoff Tool 的自动生成
# langgraph_supervisor/supervisor.py:175-186
handoff_tools = [
    create_handoff_tool(
        agent_name=agent_name,
        name=f"transfer_to_{agent_name}",   # 工具名：transfer_to_weather_expert
        add_handoff_messages=True,
    )
    for agent_name in agent_names
]

# 3. Handoff Tool 返回 Command(goto=agent_name)
# langgraph_supervisor/handoff.py:84-122
def handoff_to_agent(state, tool_call_id) -> Command:
    return Command(
        goto=agent_name,
        graph=Command.PARENT,
        update={...}
    )
""")

# ==========================================
# 使用示例（需要安装 langgraph-supervisor）
# ==========================================
print("\n使用示例:")
print("-" * 40)
print("""
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

# 创建专业化 Agent
weather_agent = create_react_agent(
    llm, tools=[query_weather], name="weather_expert"
)
order_agent = create_react_agent(
    llm, tools=[query_order, cancel_order], name="order_expert"
)
refund_agent = create_react_agent(
    llm, tools=[refund_order], name="refund_expert"
)

# 创建 Supervisor
workflow = create_supervisor(
    [weather_agent, order_agent, refund_agent],
    model=llm,
    prompt="你是调度主管，根据用户问题分发给对应专员。",
)
app = workflow.compile()
""")

print("\n完整数据流:")
print("""
用户："查北京天气"
→ START → supervisor (LLM 思考)
→ supervisor 输出 tool_calls: [transfer_to_weather_expert]
→ ToolNode 执行 handoff_to_agent("weather_expert")
→ handoff_to_agent 返回 Command(goto="weather_expert")
→ weather_expert 节点执行（完整 create_react_agent）
→ weather_expert 返回结果 → 自动回到 supervisor
→ supervisor 再次 LLM 思考 → 输出 AIMessage（无 tool_calls）→ END
""")

**Supervisor 的关键特性**：
- `output_mode` 控制 sub-agent 的输出是否带回 supervisor：`"full_history"` 带回全部历史，`"last_message"`（默认）只带回最终答案
- Supervisor 通过 `add_edge(agent.name, supervisor_agent.name)` 保证 sub-agent 执行完后自动回到 supervisor
- Handoff tool 不查天气、不查订单——它只返回 `Command(goto=agent_name)`，告诉 LangGraph 跳转到哪个 agent

### 8.2 create_swarm（群体协作模式）

来自 `langgraph-swarm`，适合**平级 Agent 动态交接**的场景。**与 Supervisor 最大的区别：没有中心调度器。**

In [ ]:
# ==========================================
# create_swarm 核心源码分析
# ==========================================

print("create_swarm 核心源码分析:")
print("=" * 60)
print("""
# 1. Swarm State：多了一个 active_agent 字段
# langgraph_swarm/swarm.py:13-19
class SwarmState(MessagesState):
    active_agent: str | None   # ← 关键：记录当前活跃的 agent

# 2. 图的构建：没有 supervisor，直接从 START 路由
# langgraph_swarm/swarm.py:270-287
builder = StateGraph(state_schema, context_schema)

# START → active_agent 的条件路由
add_active_agent_router(
    builder,
    route_to=agent_names,              # ["Alice", "Bob"]
    default_active_agent="Alice",
)

# 3. 核心路由器
# langgraph_swarm/swarm.py:163-166
def route_to_active_agent(state: dict) -> str:
    return state.get("active_agent", default_active_agent)

builder.add_conditional_edges(START, route_to_active_agent, path_map=route_to)

# 4. Handoff Tool：更新 active_agent 状态
# langgraph_swarm/handoff.py:70-90
def handoff_to_agent(state, tool_call_id) -> Command:
    return Command(
        goto=agent_name,
        graph=Command.PARENT,
        update={
            "messages": [...],
            "active_agent": agent_name,   # ← 关键：更新 active_agent！
        },
    )
""")

print("\n使用示例:")
print("-" * 40)
print("""
from langgraph_swarm import create_handoff_tool, create_swarm

alice = create_react_agent(
    llm,
    tools=[
        query_order,
        create_handoff_tool(agent_name="Bob", description="技术问题转接给 Bob"),
    ],
    name="Alice",
)

bob = create_react_agent(
    llm,
    tools=[
        query_logs,
        create_handoff_tool(agent_name="Alice", description="转回 Alice"),
    ],
    name="Bob",
)

workflow = create_swarm([alice, bob], default_active_agent="Alice")
app = workflow.compile(checkpointer=checkpointer)
""")

print("\n完整数据流:")
print("""
用户："我订单有问题"
→ START → route_to_active_agent("Alice") → Alice 节点
→ Alice 内部：LLM 思考 → 调用 query_order → 得到结果
→ Alice 内部：LLM 判断需要技术支持 → tool_calls: [transfer_to_Bob]
→ Alice 的 ToolNode 执行 handoff_to_Bob
→ handoff_to_Bob 返回 Command(
      goto="Bob",
      update={"active_agent": "Bob", "messages": [...]}
  )
→ Bob 节点执行 → 解决技术问题 → 最终回复用户
→ Bob 内部 LLM 输出无 tool_calls → 图结束

[等待用户下一次输入]
用户："谢谢"
→ START → route_to_active_agent("Bob") → Bob 节点（active_agent 还是 "Bob"）
→ Bob 回复："不客气"
""")

**Supervisor vs Swarm 关键区别**：

| 特性 | Supervisor | Swarm |
|------|-----------|-------|
| 架构 | 中心化（一个主管调度） | 去中心化（平级交接） |
| 调度器 | 有 supervisor 节点 | 无 supervisor，START 直接路由 |
| Handoff 后 | 自动回到 supervisor | 图结束，下次从 START 重新进入 |
| 状态管理 | supervisor 统一管理 | `active_agent` 字段记录当前 agent |
| 适用场景 | 任务分发、专员调度 | 客服转接、平级协作 |

### 8.3 三种模式选型总结

| 模式 | 核心函数 | 来源包 | 适用场景 |
|------|---------|--------|---------|
| 单 Agent | `create_react_agent` | `langgraph` | 一个 LLM + 多个工具 |
| 监督者 | `create_supervisor` | `langgraph-supervisor` | 一个主管调度多个专员 |
| 群体协作 | `create_swarm` | `langgraph-swarm` | 多个专员平级交接 |

## 9. 预构建 vs 自定义 StateGraph：决策框架

### 9.1 对比矩阵

```
create_react_agent（黑盒）           vs          自定义 StateGraph（白盒）
┌─────────────────────┐              ┌─────────────────────────────────────┐
│ plan → tool → observe│              │ [思考] → [决策] → [工具] → [观察]     │
│    ↑___________↓    │              │    ↑___________________________↓    │
│  （内部自动循环）      │              │  （每个节点你都可以自定义、监控、干预）  │
└─────────────────────┘              └─────────────────────────────────────┘
优点：3行代码搞定                          优点：完全可控，中间状态可见
缺点：中间状态不可见，循环控制受限            缺点：代码量多，需要理解图机制
适用：快速原型、标准 ReAct 场景              适用：复杂路由、HITL、自定义状态管理
```

In [ ]:
# ==========================================
# 维度对比表（用代码打印）
# ==========================================
comparison = [
    ("开发速度", "⭐⭐⭐ 快（3 行代码）", "⭐⭐ 中等"),
    ("可控性", "⭐⭐ 黑盒", "⭐⭐⭐ 白盒，每步可控"),
    ("可观测性", "⭐⭐ 只能看到输入输出", "⭐⭐⭐ 每节点状态可见"),
    ("复杂路由", "⭐ 不支持", "⭐⭐⭐ 任意图结构"),
    ("HITL 支持", "⭐ 弱", "⭐⭐⭐ 强"),
    ("自定义 State", "⭐ 只有 messages", "⭐⭐⭐ 任意字段"),
]

print(f"{'维度':<12} {'预构建 Agent':<25} {'自定义 StateGraph':<25}")
print("-" * 65)
for dim, prebuilt, custom in comparison:
    print(f"{dim:<12} {prebuilt:<25} {custom:<25}")

### 9.2 决策树

```
你的需求是什么？
│
├─ 标准 ReAct 循环（思考→工具→观察→再思考）？
│   ├─ 不需要看中间过程？
│   │   └─ 不需要自定义 State？
│   │       └─ 不需要复杂路由？
│   │           → create_react_agent ✅
│   │
│   └─ 需要监控每一步的 State？→ 自定义 StateGraph ✅
│
├─ 需要根据业务状态走不同分支？
│   └─ 自定义 StateGraph ✅（预构建不支持复杂路由）
│
├─ 需要人在中间审批/干预？
│   └─ 自定义 StateGraph ✅（预构建 HITL 能力弱）
│
└─ 需要自定义 State 字段（retry_count、user_role 等）？
    └─ 自定义 StateGraph ✅（预构建只有 messages 字段）
```

**关键认知**：预构建 Agent 的"快"是有代价的——你放弃了对中间过程的可见性和控制力。只要出现"需要看内部状态"或"需要干预执行"的需求，就必须切换到自定义 StateGraph。

### 9.3 实战对比：同一需求两种实现

**需求**：实现一个运维助手，能查服务器状态和执行重启命令。

#### 版本 A：create_react_agent（3 行代码）

In [ ]:
@tool
def check_cpu() -> str:
    """查询 CPU 使用率"""
    return "CPU: 45%，正常"

@tool
def restart_service(service: str) -> str:
    """重启指定服务"""
    return f"服务 {service} 已重启"


# 版本 A：3 行代码搞定
ops_tools = [check_cpu, restart_service]
ops_agent_prebuilt = create_react_agent(llm, ops_tools)

print("版本 A（预构建）创建完成！")
print("优点: 代码极简\n缺点: 无法审批，直接执行重启\n")

# 测试：直接执行了重启，没有任何阻拦
result = ops_agent_prebuilt.invoke({
    "messages": [HumanMessage(content="重启数据库服务")]
})
print(f"结果: {result['messages'][-1].content}")

#### 版本 B：自定义 StateGraph（完整可控）

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END


# ==========================================
# 自定义 State：可以扩展任意字段
# ==========================================
class OpsState(TypedDict):
    messages: Annotated[list, add]
    user_role: str           # 区分普通用户和管理员
    pending_approval: bool   # 是否有待审批操作
    execution_log: list      # 执行日志


def ops_llm_node(state: OpsState) -> dict:
    """LLM 节点：分析用户意图"""
    last_msg = state["messages"][-1].content
    
    if "重启" in last_msg:
        return {
            "pending_approval": True,
            "messages": ["检测到敏感操作：重启服务"]
        }
    elif "CPU" in last_msg or "cpu" in last_msg:
        return {
            "messages": ["CPU 状态: 45%，正常"]
        }
    return {"messages": ["未识别的操作"]}


def check_permission_node(state: OpsState) -> dict:
    """权限检查节点"""
    if state["user_role"] == "admin":
        return {
            "messages": ["管理员权限确认，允许执行"],
            "execution_log": ["admin approved"]
        }
    else:
        return {
            "messages": ["普通用户无权执行重启，已转交管理员审批"],
            "pending_approval": True
        }


def route_by_permission(state: OpsState) -> str:
    """条件路由：根据权限决定走向"""
    if state["pending_approval"] and state["user_role"] != "admin":
        return "request_approval"
    return "execute"


def execute_node(state: OpsState) -> dict:
    """执行节点"""
    return {
        "messages": ["服务已重启成功"],
        "execution_log": ["restart executed"],
        "pending_approval": False
    }


# 构建图
builder = StateGraph(OpsState)
builder.add_node("llm", ops_llm_node)
builder.add_node("check_permission", check_permission_node)
builder.add_node("execute", execute_node)
builder.add_node("request_approval", lambda s: {"messages": ["等待管理员审批..."]})

builder.add_edge(START, "llm")
builder.add_edge("llm", "check_permission")
builder.add_conditional_edges("check_permission", route_by_permission, {
    "execute": "execute",
    "request_approval": "request_approval"
})
builder.add_edge("execute", END)
builder.add_edge("request_approval", END)

ops_agent_custom = builder.compile()

print("版本 B（自定义 StateGraph）创建完成！")
print("优点: 可审批、可扩展 State、路由可控\n缺点: 代码量较大\n")

# 测试：普通用户触发审批
print("=== 普通用户请求重启 ===")
r1 = ops_agent_custom.invoke({
    "messages": [HumanMessage(content="重启数据库服务")],
    "user_role": "user",
    "pending_approval": False,
    "execution_log": []
})
print(f"结果: {r1['messages'][-1]}")
print(f"审批状态: {r1['pending_approval']}")

# 测试：管理员直接执行
print("\n=== 管理员请求重启 ===")
r2 = ops_agent_custom.invoke({
    "messages": [HumanMessage(content="重启数据库服务")],
    "user_role": "admin",
    "pending_approval": False,
    "execution_log": []
})
print(f"结果: {r2['messages'][-1]}")
print(f"执行日志: {r2['execution_log']}")

**关键差异**：
- 预构建版直接执行了 `restart_service`，没有任何阻拦
- 自定义版在 `restart_service` 前有一个条件边——检查 `user_role`，如果是普通用户，转到 `request_approval` 节点

这就是核心差异：**预构建版『快但不可控』，自定义版『慢但安全』**。

## 10. 自定义预构建 Agent

虽然 `create_react_agent` 的 State 不可扩展，但你可以通过一些技巧实现部分自定义。

### 10.1 使用 state_schema 扩展 State（有限支持）

`create_react_agent` 接受 `state_schema` 参数，允许你定义额外的 State 字段，但这些字段不会被预构建逻辑自动使用——你需要通过 `prompt` 函数或其他方式手动读取。

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage


# ==========================================
# 自定义 State Schema（扩展 messages 字段）
# ==========================================
class CustomAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    user_tier: str      # 用户等级：free / pro / enterprise
    request_count: int  # 请求计数


def custom_prompt(state: CustomAgentState) -> list[AnyMessage]:
    """
    根据用户等级动态调整 system prompt。
    虽然 state_schema 扩展了字段，但预构建逻辑不会自动使用它们，
    需要通过 prompt 函数手动读取。
    """
    tier = state.get("user_tier", "free")
    count = state.get("request_count", 0)
    
    tier_prompts = {
        "enterprise": "你是企业级专属助手，提供最高优先级的服务。",
        "pro": "你是专业版助手，提供高级功能支持。",
        "free": "你是免费版助手，提供基础服务。",
    }
    
    system_msg = SystemMessage(
        content=f"{tier_prompts.get(tier, tier_prompts['free'])} 今日请求次数: {count}"
    )
    
    return [system_msg] + list(state["messages"])


# 注意：state_schema 参数允许扩展，但预构建 Agent 的核心逻辑仍只处理 messages
# 其他字段需要你自己在 prompt 函数或外部逻辑中处理
print("自定义 State Schema 定义完成")
print("扩展字段: user_tier, request_count")
print("注意：预构建逻辑只自动处理 messages，其他字段需手动使用")

### 10.2 包装预构建 Agent 实现自定义逻辑

更实用的做法：把 `create_react_agent` 作为子图，在外层 StateGraph 中包装它，实现自定义路由和状态管理。

In [ ]:
from langgraph.graph import StateGraph, START, END


# ==========================================
# 外层 State：完全自定义
# ==========================================
class WrappedState(TypedDict):
    messages: Annotated[list, add]
    user_role: str
    audit_log: list


# 内层：预构建 Agent（只处理 messages）
inner_agent = create_react_agent(llm, tools)


def audit_node(state: WrappedState) -> dict:
    """审计节点：记录所有请求"""
    return {
        "audit_log": [f"请求来自 {state['user_role']} at {__import__('time').time()}"]
    }


def agent_wrapper(state: WrappedState) -> dict:
    """包装预构建 Agent，只传入 messages，返回结果"""
    # 预构建 Agent 只关心 messages
    result = inner_agent.invoke({"messages": state["messages"]})
    return {"messages": result["messages"]}


def permission_check(state: WrappedState) -> str:
    """权限检查：管理员放行，普通用户限制"""
    if state["user_role"] == "admin":
        return "agent"
    # 普通用户只能查天气，不能计算
    last_msg = state["messages"][-1].content
    if "计算" in last_msg or "sqrt" in last_msg:
        return "deny"
    return "agent"


def deny_node(state: WrappedState) -> dict:
    """拒绝节点"""
    return {
        "messages": [AIMessage(content="抱歉，普通用户无权使用计算功能，请升级会员。")]
    }


# 构建外层图
wrapper_builder = StateGraph(WrappedState)
wrapper_builder.add_node("audit", audit_node)
wrapper_builder.add_node("agent", agent_wrapper)
wrapper_builder.add_node("deny", deny_node)

wrapper_builder.add_edge(START, "audit")
wrapper_builder.add_edge("audit", "agent")
# 也可以加条件路由
# wrapper_builder.add_conditional_edges("audit", permission_check, {...})
wrapper_builder.add_edge("agent", END)
wrapper_builder.add_edge("deny", END)

wrapped_app = wrapper_builder.compile()

print("包装后的 Agent 创建完成！")
print("外层图管理: user_role, audit_log")
print("内层预构建 Agent 处理: messages + tools")

# 测试
r = wrapped_app.invoke({
    "messages": [HumanMessage(content="北京天气怎么样？")],
    "user_role": "admin",
    "audit_log": []
})
print(f"\n回复: {r['messages'][-1].content}")
print(f"审计日志: {r['audit_log']}")

## 11. 常见误区与注意事项

| 坑 | 现象 | 解决方案 |
|----|------|---------|
| **预构建 Agent 想加自定义 State** | 发现只能传 messages | `create_react_agent` 的 State 是固定的。需要自定义字段 → 必须用 StateGraph |
| **预构建 Agent 想改路由逻辑** | 发现只能走 LLM→Tool→LLM 循环 | 预构建的内部路由是黑盒。需要自定义分支 → 必须用 StateGraph |
| **messages 太长导致 API 报错** | 超出模型上下文限制 | 预构建 Agent 不会自动裁剪消息。需要自己加 Trimming 逻辑 |
| **把预构建用于生产环境** | 发现无法审计、无法回滚 | 预构建适合原型和内部工具。生产环境需要可观测、可干预 → 自定义 StateGraph |
| **误以为预构建性能更好** | 其实和自定义差不多 | 预构建底层也是 StateGraph，只是帮你搭好了。性能没有本质差异 |

### 调试技巧
1. **看内部过程**：用 `stream()` 代替 `invoke()`，可以看到每次 tool 调用
2. **Message 长度监控**：打印 `len(messages)` 和 `count_tokens(messages)`，及时发现膨胀
3. **Langfuse 追踪**：预构建 Agent 也支持 CallbackHandler，可以看到完整的 Tool 调用链路
4. **迁移路径**：从预构建迁移到自定义时，先 copy 预构建的 State 结构，再逐步添加自定义字段

## 12. 阶段小结

### 核心要点回顾

1. **`create_react_agent` 是单 Agent 唯一官方推荐**：LangGraph 1.x 统一用这一个，通吃所有模型
2. **多 Agent 生态扩展**：`create_supervisor`（监督者调度）、`create_swarm`（平级交接），需额外安装
3. **内部机制**：LLM → Tool → LLM 自动循环，消息历史自动管理
4. **State 限制**：只有 `messages` 一个字段，不可扩展
5. **扩展能力**：支持 system prompt、动态 prompt 函数、checkpointer 记忆、消息裁剪
6. **选型原则**：标准 ReAct + 不需要监控 → 预构建；复杂路由/HITL/自定义 State → 自定义
7. **上下文管理**：预构建不会自动裁剪，需要自己加 Trimming/Summarization/Tool 精简
8. **自定义策略**：需要自定义 State 时，用外层 StateGraph 包装预构建 Agent
9. **迁移策略**：先用预构建验证需求，确认后再迁移到自定义 StateGraph

### 记忆口诀

> **"create_react_agent 是正统，预构建求快三行搞定，自定义求控全程透明。先用预构建验证，再上 StateGraph 生产。"**

### 三种模式速查

| 模式 | 函数 | 包 | 核心特点 |
|------|------|-----|---------|
| 单 Agent | `create_react_agent` | `langgraph` | 3行代码，自动 ReAct 循环 |
| 监督者 | `create_supervisor` | `langgraph-supervisor` | 中心化调度，handoff 后回 supervisor |
| 群体协作 | `create_swarm` | `langgraph-swarm` | 去中心化，`active_agent` 状态路由 |

## 13. 课后练习

1. **工具组合练习**：用 `create_react_agent` 搭建一个能查天气、查汇率、做单位换算的旅行助手
2. **消息裁剪实战**：实现一个带消息自动裁剪的 Agent，当历史超过 20 条时自动摘要
3. **多 Agent 设计**：设计一个客服场景，用 `create_supervisor` 实现「问题分类 → 专员处理 → 汇总回复」的流程
4. **选型判断**：给出 3 个业务场景，判断用预构建还是自定义 StateGraph
   - "内部客服机器人，只需要查知识库" → ?
   - "银行转账助手，涉及金额需要审批" → ?
   - "快速原型验证，3 天内出 demo" → ?
5. **包装练习**：用外层 StateGraph 包装 `create_react_agent`，实现用户等级限制（免费用户每天只能调用 5 次）

---

**下一节**: LG-04 人机协作与 Hooks